# 🧪 Computational Thermodynamics — Binary Eutectic System

A CALPHAD interactive tool using **pycalphad**.

| Feature | Details |
|---------|---------|
| Free-energy curves | G–x plots with **common-tangent construction** |
| Phase diagram | Binary T–x diagram with live temperature indicator |
| Thermo properties | Enthalpy · Entropy · Activity |

> **No installation needed** — everything runs here on Google Colab.
> Just press **Runtime → Run all** and follow the widgets below.

In [ ]:
# @title ⚙️ Step 1 — Install packages (run once per session)
import sys
!{sys.executable} -m pip install pycalphad scipy matplotlib -q
print("✅ Packages ready!")

In [ ]:
# @title 📦 Step 2 — Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import warnings, io
warnings.filterwarnings('ignore')

from pycalphad import Database, binplot, calculate, equilibrium, variables as v
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'lines.linewidth': 2.0, 'figure.dpi': 110,
})
print("✅ Imports done!")

In [ ]:
# @title 🗄️ Step 3 — Embedded databases
AGCU_TDB = """
$ Synthesized Ag-Cu Eutectic System
ELEMENT AG STANDARD  107.868 4260.32 42.55 !
ELEMENT CU STANDARD   63.546 3344.0  33.15 !
FUNCTION UN_ASS  298.15 0.0; 3000.0 N !
PHASE LIQUID:L %  1  1.0  !
CONSTITUENT LIQUID:L :AG,CU :  !
PARAMETER G(LIQUID,AG;0)  298.15  +11500-9.3*T; 3000 N !
PARAMETER G(LIQUID,CU;0)  298.15  +13000-9.6*T; 3000 N !
PARAMETER G(LIQUID,AG,CU;0)  298.15  +12000-3.0*T; 3000 N !
PHASE FCC_A1 %  2  1.0  1.0  !
CONSTITUENT FCC_A1 :AG,CU : VA : !
PARAMETER G(FCC_A1,AG:VA;0) 298.15  0.0; 3000 N !
PARAMETER G(FCC_A1,CU:VA;0) 298.15  0.0; 3000 N !
PARAMETER G(FCC_A1,AG,CU:VA;0) 298.15 +28000-4.0*T; 3000 N !
"""

ALZN_TDB = "$ ALZN\n$\n$ TDB-file for the thermodynamic assessment of the Al-ZN system\n$\n$-------------------------------------------------------------------------------\n$ 2011.11.9\n$ \n$ TDB file created by T.Abe, K.Hashimoto and Y.sawada\n$\n$ Particle Simulation and Thermodynamics Group, National Institute for \n$ Materials Science. 1-2-1 Sengen, Tsukuba, Ibaraki 305-0047, Japan\n$ e-mail: abe.taichi@nims.go.jp\n$ Copyright (C) NIMS 2009\n$\n$ ------------------------------------------------------------------------------\n$ PARAMETERS ARE TAKEN FROM \n$ Reevaluation of the Al-Zn System,\n$ Sabine an Mey, Z.Metallkd., 84 (1993) 451-455.\n$\n$ ------------------------------------------------------------------------------\n\n ELEMENT /-   ELECTRON_GAS              0.0000E+00  0.0000E+00  0.0000E+00!\n ELEMENT VA   VACUUM                    0.0000E+00  0.0000E+00  0.0000E+00!\n ELEMENT AL   FCC_A1                    2.6982E+01  4.5773E+03  2.8322E+01!\n ELEMENT ZN   HCP_ZN                    6.5390E+01  5.6568E+03  4.1631E+01!\n\n$-------------------------------------------------------------------------------\n$ FUNCTIONS FOR PURE AND OTHERS\n$-------------------------------------------------------------------------------\n FUNCTION  GHSERAL  298.0\n    -7976.15+137.0715*T-24.36720*T*LN(T)-1.884662E-3*T**2-0.877664E-6*T**3\n    +74092*T**(-1);                                                   700.00 Y\n    -11276.24+223.0269*T-38.58443*T*LN(T)+18.531982E-3*T**2-5.764227E-6*T**3\n    +74092*T**(-1);                                                    933.6 Y\n    -11277.68+188.6620*T-31.74819*T*LN(T)-1234.26E25*T**(-9);        2900.00 N ! \n FUNCTION  GALLIQ   298.0\n    +3029.403+125.2307*T-24.36720*T*LN(T)-1.884662E-3*T**2-0.877664E-6*T**3\n    +74092*T**(-1)+79.401E-21*T**7;                                   700.00 Y\n    -270.6860+211.1861*T-38.58443*T*LN(T)+18.53198E-3*T**2-5.764227E-6*T**3\n    +74092*T**(-1)+79.401E-21*T**7;                                    933.6 Y\n    -795.7090+177.4100*T-31.74819*T*LN(T);                           2900.00 N !\n FUNCTION  GALHCP   298.0  +5481-1.8*T+GHSERAL#;                      6000  N !\n\n\n FUNCTION GHSERZN    298.0  -7285.787+118.4693*T-23.70131*T*LN(T)\n     -.001712034*T**2-1.264963E-06*T**3;                              692.7 Y\n     -11070.60+172.3449*T-31.38*T*LN(T)+4.70657E+26*T**(-9);           1700 N !\n $FUNCTION GZNLIQ     298.0  -1.285170+108.1769*T-23.70131*T*LN(T)\n $    -.001712034*T**2-1.264963E-06*T**3-3.585652E-19*T**7;            692.7 Y\n $    -11070.60+172.3449*T-31.38*T*LN(T)+4.70657E+26*T**(-9);           1700 N !\n FUNCTION GZNLIQ     298.14  +7157.213-10.29299*T-3.5896E-19*T**7+GHSERZN#;\n                                                                      692.7 Y\n     +7450.168-10.737066*T-4.7051E+26*T**(-9)+GHSERZN#;                 1700 N !\n FUNCTION GZNFCC     298.15  +2969.82-1.56968*T+GHSERZN#;               1700 N !\n\n$-------------------------------------------------------------------------------\n TYPE_DEFINITION % SEQ *!\n DEFINE_SYSTEM_DEFAULT ELEMENT 2 !\n DEFAULT_COMMAND DEF_SYS_ELEMENT VA /- !\n\n$-------------------------------------------------------------------------------\n$ PARAMETERS FOR LIQUID PHASE\n$-------------------------------------------------------------------------------\n PHASE LIQUID %  1  1.0  !\n CONSTITUENT LIQUID :AL,ZN :  !\n   PARAMETER G(LIQUID,AL;0)      298.15  +GALLIQ#;                      2900 N !\n   PARAMETER G(LIQUID,ZN;0)      298.15  +GZNLIQ#;                      1700 N !\n   PARAMETER G(LIQUID,AL,ZN;0)   298.15  +10465.5-3.39259*T;            6000 N !\n\n$-------------------------------------------------------------------------------\n$ FUNCTIONS FOR FCC_A1\n$-------------------------------------------------------------------------------\n PHASE FCC_A1  %  1  1.0  !\n CONSTITUENT FCC_A1  :AL,ZN :  !\n   PARAMETER G(FCC_A1,AL;0)      298.15  +GHSERAL#;                     2900 N !\n   PARAMETER G(FCC_A1,ZN;0)      298.15  +GZNFCC#;                      1700 N !\n   PARAMETER G(FCC_A1,AL,ZN;0)   298.15  +7297.5+0.47512*T;             6000 N !\n   PARAMETER G(FCC_A1,AL,ZN;1)   298.15  +6612.9-4.5911*T;              6000 N !\n   PARAMETER G(FCC_A1,AL,ZN;2)   298.15  -3097.2+3.30635*T;             6000 N !\n\n$-------------------------------------------------------------------------------\n$ FUNCTIONS FOR HCP_A3\n$-------------------------------------------------------------------------------\n PHASE HCP_A3  %  1  1.0  !\n CONSTITUENT HCP_A3  :AL,ZN :  !\n  PARAMETER G(HCP_A3,AL;0)       298.15  +GALHCP#;                      2900 N !\n  PARAMETER G(HCP_A3,ZN;0)       298.15  +GHSERZN#;                     1700 N !\n  PARAMETER G(HCP_A3,AL,ZN;0)    298.15  +18821.0-8.95255*T;            6000 N !\n  PARAMETER G(HCP_A3,AL,ZN;3)    298.15  -702.8;                        6000 N !\n\n$\n$------------------------------------------------------------------- END OF LINE\n\n"

DATABASES = {
    "Ag-Cu (Synthetic Eutectic)": (AGCU_TDB, "CU", 800, 1500),
    "Al-Zn (Mey 1993)":          (ALZN_TDB, "ZN", 500, 1100),
}
print("✅ Databases embedded!")

In [ ]:
# @title 🔧 Step 4 — Common-tangent functions

def _compute_common_tangents(x_liq, g_liq, x_fcc, g_fcc):
    """Return list of (x1,g1,x2,g2) tangent contact pairs between two curves."""
    dx = 1e-4
    mask_l, mask_f = np.isfinite(g_liq), np.isfinite(g_fcc)
    xl, gl = x_liq[mask_l], g_liq[mask_l]
    xf, gf = x_fcc[mask_f], g_fcc[mask_f]
    if len(xl) < 4 or len(xf) < 4:
        return []
    il = interp1d(xl, gl, kind='cubic', bounds_error=False, fill_value=np.nan)
    if_ = interp1d(xf, gf, kind='cubic', bounds_error=False, fill_value=np.nan)

    def obj(p):
        xa, xb = p
        if xa <= xl.min()+0.005 or xb >= xf.max()-0.005 or xb - xa < 0.02:
            return 1e10
        ga, gb = il(xa), if_(xb)
        if np.isnan(ga) or np.isnan(gb): return 1e10
        sec = (gb - ga) / (xb - xa + 1e-12)
        dga = (il(xa+dx) - il(xa-dx)) / (2*dx)
        dgb = (if_(xb+dx) - if_(xb-dx)) / (2*dx)
        return (dga - sec)**2 + (dgb - sec)**2

    seen, tangents = [], []
    for xa0 in np.linspace(xl.min()+0.05, xl.max()-0.05, 12):
        for xb0 in np.linspace(xf.min()+0.05, xf.max()-0.05, 12):
            if xb0 <= xa0: continue
            res = minimize(obj, [xa0, xb0], method='Nelder-Mead',
                           options={'xatol':1e-5,'fatol':1e-5,'maxiter':1000})
            if res.fun > 1e-3: continue
            xa_r, xb_r = res.x
            if not (xl.min()<xa_r<xl.max() and xf.min()<xb_r<xf.max()): continue
            if xa_r >= xb_r - 0.01: continue
            if any(abs(xa_r-s[0])<0.04 and abs(xb_r-s[1])<0.04 for s in seen): continue
            ga_r, gb_r = float(il(xa_r)), float(if_(xb_r))
            if np.isfinite(ga_r) and np.isfinite(gb_r):
                seen.append((xa_r, xb_r))
                tangents.append((xa_r, ga_r, xb_r, gb_r))
    return tangents


def _find_fcc_fcc_tangent(xf, gf):
    """Find common tangent inside an FCC miscibility gap."""
    mask = np.isfinite(gf)
    xf, gf = xf[mask], gf[mask]
    if len(xf) < 6: return None
    dx = 1e-4
    interp = interp1d(xf, gf, kind='cubic', bounds_error=False, fill_value=np.nan)

    def obj(p):
        xa, xb = p
        if xa<=0.01 or xb>=0.99 or xb-xa<0.05: return 1e10
        ga, gb = interp(xa), interp(xb)
        if np.isnan(ga) or np.isnan(gb): return 1e10
        sec = (gb-ga)/(xb-xa)
        dga = (interp(xa+dx)-interp(xa-dx))/(2*dx)
        dgb = (interp(xb+dx)-interp(xb-dx))/(2*dx)
        return (dga-sec)**2 + (dgb-sec)**2 + (dga-dgb)**2

    best = None
    for xa0 in np.linspace(0.05, 0.45, 7):
        for xb0 in np.linspace(0.55, 0.95, 7):
            res = minimize(obj, [xa0, xb0], method='Nelder-Mead',
                           options={'xatol':1e-5,'fatol':1e-5,'maxiter':2000})
            if res.fun < 1e-3:
                xa_r, xb_r = res.x
                if 0 < xa_r < xb_r < 1 and xb_r - xa_r > 0.05:
                    ga_r, gb_r = float(interp(xa_r)), float(interp(xb_r))
                    if np.isfinite(ga_r) and np.isfinite(gb_r):
                        if best is None or res.fun < best[4]:
                            best = (xa_r, ga_r, xb_r, gb_r, res.fun)
    return best[:4] if best else None

print("✅ Helper functions defined!")

In [ ]:
# @title 🗃️ Step 5 — Select database

STATE = {}   # will hold: dbf, comps, phases, second_el, T_min, T_max, label

def _load_state(tdb_str, second_el, T_min, T_max, label):
    dbf = Database(tdb_str)
    all_comps = sorted(c for c in dbf.elements if c not in ('VA','/-'))
    comps  = all_comps + ['VA']
    phases = list(dbf.phases.keys())
    STATE.update(dict(dbf=dbf, comps=comps, phases=phases,
                      second_el=second_el, T_min=T_min, T_max=T_max, label=label))
    print(f"  ✔  Loaded: {label}")
    print(f"     Components : {comps}")
    print(f"     Phases     : {phases}")

db_dropdown = widgets.Dropdown(
    options=list(DATABASES.keys()) + ['📁 Upload custom .tdb'],
    description='Database:', style={'description_width':'initial'})

upload_widget = widgets.FileUpload(
    accept='.tdb', multiple=False,
    description='Upload .tdb', layout=widgets.Layout(display='none'))

load_btn = widgets.Button(description='Load Database', button_style='success',
                          icon='database')
out_load = widgets.Output()

def on_dropdown(change):
    upload_widget.layout.display = 'flex' if change['new'].startswith('📁') else 'none'
db_dropdown.observe(on_dropdown, names='value')

def on_load(_):
    with out_load:
        clear_output(wait=True)
        choice = db_dropdown.value
        if choice.startswith('📁'):
            if not upload_widget.value:
                print("⚠️  Please upload a .tdb file first.")
                return
            uploaded = list(upload_widget.value.values())[0]
            tdb_str  = uploaded['content'].decode('utf-8')
            # Probe elements
            dbf_tmp  = Database(tdb_str)
            els = sorted(c for c in dbf_tmp.elements if c not in ('VA','/-'))
            sel = els[1] if len(els) >= 2 else els[0]
            _load_state(tdb_str, sel, 500, 1600, "Custom .tdb")
        else:
            tdb_str, second_el, T_min, T_max = DATABASES[choice]
            _load_state(tdb_str, second_el, T_min, T_max, choice)

load_btn.on_click(on_load)
display(widgets.VBox([db_dropdown, upload_widget, load_btn, out_load]))

# Auto-load default
_load_state(*DATABASES["Ag-Cu (Synthetic Eutectic)"], "Ag-Cu (Synthetic Eutectic)")

## 1. Binary Phase Diagram

The dashed gold line shows the currently selected temperature — it updates live with the slider below.

In [ ]:
# @title 📊 Phase Diagram (computed once — may take ~30 s)
print("Computing phase diagram … please wait.")
dbf  = STATE['dbf'];  comps = STATE['comps'];  phases = STATE['phases']
sel  = STATE['second_el']
T_mn = STATE['T_min']; T_mx = STATE['T_max']

fig_pd, ax_pd = plt.subplots(figsize=(8, 5))
try:
    binplot(dbf, comps, phases,
            {v.X(sel): (0, 1, 0.02), v.T: (T_mn, T_mx, 10), v.P: 101325},
            ax=ax_pd)
except Exception as e:
    ax_pd.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
               transform=ax_pd.transAxes, color='red')
ax_pd.set_title(f'Binary Phase Diagram — {STATE["label"]}', fontweight='bold')
ax_pd.set_xlabel(f'Mole Fraction {sel}')
ax_pd.set_ylabel('Temperature (K)')
ax_pd.set_ylim(T_mn, T_mx)
plt.tight_layout()
plt.show()
STATE['fig_pd'] = fig_pd
STATE['ax_pd']  = ax_pd
print("✅ Phase diagram ready!")

## 2. Gibbs Free Energy Curves

Move the slider to change temperature. The **dashed black lines** are the common tangents — their endpoints (filled circles) give the equilibrium compositions. The gold dashed line on the phase diagram above tracks your selected temperature.

In [ ]:
# @title ⚡ Free Energy Curves — Interactive
_COLORS = {'LIQUID':'#2196F3','FCC_A1':'#F44336','HCP_A3':'#4CAF50',
           'BCC_A2':'#FF9800','HCP_ZN':'#9C27B0'}
_temp_line = [None]

out_ge = widgets.Output()

def draw_free_energy(T):
    with out_ge:
        clear_output(wait=True)
        dbf  = STATE['dbf'];  comps = STATE['comps'];  phases = STATE['phases']
        sel  = STATE['second_el']

        fig, ax = plt.subplots(figsize=(9, 5))
        x_data, g_data, c_map = {}, {}, {}
        cidx = 0
        default_cols = plt.cm.tab10.colors

        for ph in phases:
            try:
                res = calculate(dbf, comps, ph, P=101325, T=T,
                                output='GM', points_per_phase=200)
                xv = res.X.sel(component=sel).values.flatten()
                gv = res.GM.values.flatten()
                order = np.argsort(xv); xv, gv = xv[order], gv[order]
                mask = np.isfinite(gv)
                if mask.sum() < 3: continue
                x_data[ph] = xv; g_data[ph] = gv
                col = _COLORS.get(ph, default_cols[cidx % len(default_cols)])
                c_map[ph] = col; cidx += 1
                ax.plot(xv[mask], gv[mask], color=col, label=ph, zorder=3)
            except Exception:
                pass

        liq = [p for p in x_data if 'LIQUID' in p.upper()]
        sol = [p for p in x_data if 'LIQUID' not in p.upper()]
        first_tan = True
        for lp in liq:
            for sp in sol:
                for (x1,g1,x2,g2) in _compute_common_tangents(
                        x_data[lp], g_data[lp], x_data[sp], g_data[sp]):
                    lbl = 'Common tangent' if first_tan else ''
                    ax.plot([x1,x2],[g1,g2],'k--',lw=1.8,zorder=5,label=lbl)
                    ax.plot([x1,x2],[g1,g2],'ko',ms=7,zorder=6)
                    first_tan = False

        for sp in sol:
            res = _find_fcc_fcc_tangent(x_data[sp], g_data[sp])
            if res:
                x1,g1,x2,g2 = res
                ax.plot([x1,x2],[g1,g2],color=c_map[sp],ls='--',lw=1.8,zorder=5,
                        label=f'{sp} gap')
                ax.plot([x1,x2],[g1,g2],'o',color=c_map[sp],ms=7,zorder=6)

        ax.set_title(f'Molar Gibbs Free Energy  —  T = {T:.0f} K', fontweight='bold')
        ax.set_xlabel(f'Mole Fraction {sel}  ($X_{{{sel}}}$)')
        ax.set_ylabel(r'$G_m$ (J/mol)')
        ax.set_xlim(-0.02, 1.02)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
        plt.tight_layout(); plt.show()

    # ── Update temperature line on phase diagram ──────────────────────────
    ax_pd = STATE.get('ax_pd')
    if ax_pd is not None:
        if _temp_line[0] is not None:
            try: _temp_line[0].remove()
            except Exception: pass
        _temp_line[0] = ax_pd.axhline(y=T, color='gold', lw=2.2,
                                       ls='--', zorder=10, label=f'T = {T:.0f} K')
        handles, labels = ax_pd.get_legend_handles_labels()
        ax_pd.legend(handles[-3:], labels[-3:], fontsize=8, loc='upper right')
        STATE['fig_pd'].canvas.draw_idle()

T_mn = STATE['T_min']; T_mx = STATE['T_max']
T_init = round((T_mn + T_mx) / 2 / 25) * 25

sl_ge = widgets.FloatSlider(value=T_init, min=T_mn, max=T_mx, step=25,
                             description='T (K):', continuous_update=False,
                             layout=widgets.Layout(width='70%'))
display(sl_ge, out_ge)
draw_free_energy(T_init)
sl_ge.observe(lambda c: draw_free_energy(c['new']), names='value')

## 3. Thermodynamic Properties

Enthalpy · Entropy · Activity of the second component vs. composition.

In [ ]:
# @title 🌡️ Thermo Properties — Interactive
out_tp = widgets.Output()

def draw_thermo(T):
    with out_tp:
        clear_output(wait=True)
        dbf   = STATE['dbf'];  comps = STATE['comps'];  phases = STATE['phases']
        sel   = STATE['second_el']
        conds = {v.X(sel): (0.01, 0.99, 0.02), v.T: T, v.P: 101325}

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f'Thermodynamic Properties — T = {T:.0f} K', fontweight='bold')

        try:
            eq  = equilibrium(dbf, comps, phases, conds)
            xc  = eq.X.sel(component=sel).values.flatten()
            Hm  = eq.HM.values.flatten()
            Sm  = eq.SM.values.flatten()
            mu  = eq.MU.sel(component=sel).values.flatten()

            sol = [p for p in phases if 'LIQUID' not in p.upper()]
            ref_ph = sol[0] if sol else phases[0]
            ref_eq = equilibrium(dbf, comps, [ref_ph],
                                 {v.X(sel):1.0, v.T:T, v.P:101325})
            mu_ref = float(ref_eq.MU.sel(component=sel).values.flatten()[0])
            act = np.exp((mu - mu_ref) / (8.314 * T))

            axes[0].plot(xc, Hm, color='#9C27B0')
            axes[1].plot(xc, Sm, color='#009688')
            axes[2].plot(xc, act, color='#FF5722', label='Calculated')
            axes[2].plot([0,1],[0,1],'k--', label="Ideal (Raoult's)")
            axes[2].legend(fontsize=9); axes[2].set_ylim(-0.05, 1.05)
        except Exception as e:
            for ax in axes:
                ax.text(0.5, 0.5, f'Error:\n{e}', ha='center', va='center',
                        transform=ax.transAxes, color='red', fontsize=9)

        for ax, title, ylabel in zip(axes,
            ['Enthalpy of Mixing','Entropy of Mixing',f'Activity of {sel}'],
            [r'$H_m$ (J/mol)', r'$S_m$ (J/K·mol)', f'$a_{{{sel}}}$']):
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel(f'$X_{{{sel}}}$'); ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.25)

        plt.tight_layout(); plt.show()

sl_tp = widgets.FloatSlider(value=T_init, min=T_mn, max=T_mx, step=50,
                             description='T (K):', continuous_update=False,
                             layout=widgets.Layout(width='70%'))
display(sl_tp, out_tp)
draw_thermo(T_init)
sl_tp.observe(lambda c: draw_thermo(c['new']), names='value')